In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy
%pip install xgboost

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import gc

In [4]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [5]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def log_ratio_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    total = np.sum(counts)
    weights = np.log(total / counts)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def cost_sensitive_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)
    weights = max_count / counts
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [6]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [7]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [8]:
import joblib
X_train_none_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/X_train_none.joblib")
y_train_none_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/y_train_none.joblib")

X_train_smote_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/X_train_smote.joblib")
y_train_smote_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/y_train_smote.joblib")

X_train_smote_enn_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/X_train_smote_enn.joblib")
y_train_smote_enn_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped_2 = joblib.load("Sets_Oversampling_Grouped_2/y_train_smote_tomek.joblib")

X_val_imp_grouped_2 = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped_2 = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped_2 = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped_2 = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [9]:
train_datasets_grouped_2 = {
    'none': (X_train_none_grouped_2, y_train_none_grouped_2.values.ravel() if isinstance(y_train_none_grouped_2, pd.DataFrame) else y_train_none_grouped_2.ravel()),
    'smote': (X_train_smote_grouped_2, y_train_smote_grouped_2.values.ravel() if isinstance(y_train_smote_grouped_2, pd.DataFrame) else y_train_smote_grouped_2.ravel()),
    'smote_enn': (X_train_smote_enn_grouped_2, y_train_smote_enn_grouped_2.values.ravel() if isinstance(y_train_smote_enn_grouped_2, pd.DataFrame) else y_train_smote_enn_grouped_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped_2, y_train_smote_tomek_grouped_2.values.ravel() if isinstance(y_train_smote_tomek_grouped_2, pd.DataFrame) else y_train_smote_tomek_grouped_2.ravel())
}

In [10]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [11]:
def save_confusion_matrix_mlp(folder, y_true, y_pred, trial_num, dataset_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {dataset_name} Trial {trial_num}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f"{folder}/{dataset_name}_trial_{trial_num}_cm.png")
    plt.close()

In [12]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate):
        super().__init__()
        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(nn.LeakyReLU(0.01))
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        layers.append(nn.Linear(in_f, 13))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

In [ ]:
import os
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")
device = torch.device("cuda")

log_folder = "Logs_MLP_Baseline_1"
os.makedirs(log_folder, exist_ok=True)

print("Prepando tensores en VRAM...")
X_val_vram = torch.tensor(np.array(X_val_imp_grouped_2, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_grouped_2, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

train_tensors = {}
for ds_name in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    x_raw, y_raw = train_datasets_grouped_2[ds_name]
    train_tensors[ds_name] = (
        torch.tensor(np.array(x_raw, dtype=np.float32)).to(device),
        torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)
    )

def objective(trial):
    x_train, y_train = train_tensors[current_dataset]
    
    batch_size = trial.suggest_categorical('batch_size', [4096, 8192])
    n_layers = trial.suggest_int('n_layers', 2, 3)
    n_units = trial.suggest_int('n_units', 512, 1024, step=256)
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    
    model = TabularMLP(x_train.shape[1], n_layers, n_units, 0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    num_samples = x_train.shape[0]
    for epoch in range(50):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = x_train[indices], y_train[indices]
            optimizer.zero_grad()
            loss = F.cross_entropy(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_val_vram)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        f1 = f1_score(y_val_cpu, preds, average='macro')
    
    save_confusion_matrix_mlp(log_folder, y_val_cpu, preds, trial.number, current_dataset)
    report = classification_report(y_val_cpu, preds, zero_division=0)
    
    with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", "w") as f:
        f.write(f"Trial {trial.number}\n")
        f.write(f"F1 Macro: {f1:.4f}\n")
        f.write(f"Hiperparametros: {trial.params}\n\n")
        f.write(f"Reporte de Clasificacion:\n{report}")
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return f1

resultados_globales = {}

for current_dataset in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    print(f"\nEntrenando {current_dataset}...")
    study_name=f"study_mlp_{current_dataset}"
    study = optuna.create_study(direction='maximize',study_name=study_name)
    study.optimize(objective, n_trials=25)
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params
    }
    
    best_log_msg = f"""Mejor trial para dataset: {current_dataset}
Trial numero: {study.best_trial.number}
F1 macro alcanzado: {study.best_value:.4f}
Hiperparametros:
{study.best_params}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)
    
    print(f"Mejor F1 en {current_dataset}: {study.best_value:.4f}")

informe_final = "Informe final: Baseline MLP\n"

for dataset, metricas in resultados_globales.items():
    informe_final += f"Dataset: {dataset}\n"
    informe_final += f"- Mejor trial: {metricas['best_trial']}\n"
    informe_final += f"- F1 macro alcanzado: {metricas['best_f1_macro']:.4f}\n"
    informe_final += f"- Hiperparametros ganadores:\n"
    for param, value in metricas['best_params'].items():
        informe_final += f"  * {param}: {value}\n"
    informe_final += "\n"

ganador_absoluto = max(resultados_globales, key=lambda x: resultados_globales[x]['best_f1_macro'])
informe_final += f"Ganador absoluto: {ganador_absoluto} (F1 macro: {resultados_globales[ganador_absoluto]['best_f1_macro']:.4f})\n"

print("\n" + informe_final)

with open(f"{log_folder}/resumen_global_resultados.txt", 'w') as f:
    f.write(informe_final)

print(f"Proceso finalizado. Todo guardado en {log_folder}")

Prepando tensores en VRAM...


[I 2026-03-18 20:52:59,362] A new study created in memory with name: study_mlp_none



Entrenando none...


[I 2026-03-18 20:54:23,874] Trial 0 finished with value: 0.8789727477329053 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 1024, 'lr': 0.0006179044751881685}. Best is trial 0 with value: 0.8789727477329053.
[I 2026-03-18 20:56:05,210] Trial 1 finished with value: 0.867296619164816 and parameters: {'batch_size': 4096, 'n_layers': 3, 'n_units': 512, 'lr': 0.0007436829110428842}. Best is trial 0 with value: 0.8789727477329053.
[I 2026-03-18 20:57:47,445] Trial 2 finished with value: 0.7852588682904714 and parameters: {'batch_size': 4096, 'n_layers': 3, 'n_units': 512, 'lr': 0.00011145509101145481}. Best is trial 0 with value: 0.8789727477329053.
[I 2026-03-18 20:58:40,573] Trial 3 finished with value: 0.7771679502577962 and parameters: {'batch_size': 8192, 'n_layers': 3, 'n_units': 1024, 'lr': 0.00018033533559773163}. Best is trial 0 with value: 0.8789727477329053.
[I 2026-03-18 21:00:05,483] Trial 4 finished with value: 0.8396135162956931 and parameters: {'batch_size': 40

Mejor F1 en none: 0.8798

Entrenando smote_tomek...


...

Mejor F1 en smote_tomek: 0.8848

Entrenando smote...


[I 2026-03-18 21:51:19,766] Trial 0 finished with value: 0.8801111072434774 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 1024, 'lr': 0.0006861019972566098}. Best is trial 0 with value: 0.8801111072434774.
[I 2026-03-18 21:52:06,897] Trial 1 finished with value: 0.8585816086919785 and parameters: {'batch_size': 8192, 'n_layers': 2, 'n_units': 512, 'lr': 0.00033850293122954646}. Best is trial 0 with value: 0.8801111072434774.
[I 2026-03-18 21:53:01,752] Trial 2 finished with value: 0.8693357633604355 and parameters: {'batch_size': 8192, 'n_layers': 3, 'n_units': 512, 'lr': 0.0001131744043875548}. Best is trial 0 with value: 0.8801111072434774.
[I 2026-03-18 21:54:47,855] Trial 3 finished with value: 0.8666589116428126 and parameters: {'batch_size': 4096, 'n_layers': 3, 'n_units': 768, 'lr': 0.0007432592732516806}. Best is trial 0 with value: 0.8801111072434774.
[I 2026-03-18 21:56:31,484] Trial 4 finished with value: 0.8282496961167192 and parameters: {'batch_size': 409

Mejor F1 en smote: 0.8858

Entrenando smote_enn...


[I 2026-03-18 22:21:55,849] Trial 0 finished with value: 0.8152915583510258 and parameters: {'batch_size': 4096, 'n_layers': 3, 'n_units': 512, 'lr': 0.00013090265085930118}. Best is trial 0 with value: 0.8152915583510258.
[I 2026-03-18 22:23:22,852] Trial 1 finished with value: 0.7913465270601591 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 768, 'lr': 0.0009862414853993632}. Best is trial 0 with value: 0.8152915583510258.
[I 2026-03-18 22:25:06,509] Trial 2 finished with value: 0.8167166358474586 and parameters: {'batch_size': 4096, 'n_layers': 3, 'n_units': 1024, 'lr': 0.0006016908117641884}. Best is trial 2 with value: 0.8167166358474586.
[I 2026-03-18 22:25:55,196] Trial 3 finished with value: 0.7977022866622383 and parameters: {'batch_size': 8192, 'n_layers': 2, 'n_units': 1024, 'lr': 0.0004476270457816368}. Best is trial 2 with value: 0.8167166358474586.
[I 2026-03-18 22:26:43,105] Trial 4 finished with value: 0.8162793840739057 and parameters: {'batch_size': 81

Mejor F1 en smote_enn: 0.8180

Informe final: Baseline MLP
Dataset: none
- Mejor trial: 21
- F1 macro alcanzado: 0.8798
- Hiperparametros ganadores:
  * batch_size: 4096
  * n_layers: 2
  * n_units: 1024
  * lr: 0.0006200093358476453

Dataset: smote_tomek
- Mejor trial: 15
- F1 macro alcanzado: 0.8848
- Hiperparametros ganadores:
  * batch_size: 8192
  * n_layers: 3
  * n_units: 768
  * lr: 0.0009753865891732886

Dataset: smote
- Mejor trial: 23
- F1 macro alcanzado: 0.8858
- Hiperparametros ganadores:
  * batch_size: 8192
  * n_layers: 2
  * n_units: 1024
  * lr: 0.00044931955061787595

Dataset: smote_enn
- Mejor trial: 24
- F1 macro alcanzado: 0.8180
- Hiperparametros ganadores:
  * batch_size: 4096
  * n_layers: 3
  * n_units: 768
  * lr: 0.0005231693596663353

Ganador absoluto: smote (F1 macro: 0.8858)

Proceso finalizado. Todo guardado en Logs_MLP_Baseline_1


In [ ]:
import os
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")
device = torch.device("cuda")

log_folder = "Logs_MLP_Baseline_2"
os.makedirs(log_folder, exist_ok=True)

X_val_vram = torch.tensor(np.array(X_val_imp_grouped_2, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_grouped_2, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

train_tensors = {}
for ds_name in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    x_raw, y_raw = train_datasets_grouped_2[ds_name]
    train_tensors[ds_name] = (
        torch.tensor(np.array(x_raw, dtype=np.float32)).to(device),
        torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)
    )

def objective(trial):
    x_train, y_train = train_tensors[current_dataset]
    
    batch_size = trial.suggest_categorical('batch_size', [4096, 8192])
    n_layers = trial.suggest_int('n_layers', 2, 3)
    n_units = trial.suggest_int('n_units', 512, 1024, step=128)
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    
    gamma_opt = trial.suggest_float('gamma', 1.5, 4.0)
    
    model = TabularMLP(x_train.shape[1], n_layers, n_units, 0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = FocalLoss(gamma=gamma_opt)

    num_samples = x_train.shape[0]
    for epoch in range(50):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = x_train[indices], y_train[indices]
            
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_val_vram)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        f1 = f1_score(y_val_cpu, preds, average='macro')
    
    save_confusion_matrix_mlp(log_folder, y_val_cpu, preds, trial.number, current_dataset)
    report = classification_report(y_val_cpu, preds, zero_division=0)
    with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", "w") as f:
        f.write(f"F1 Macro: {f1:.4f}\nParams: {trial.params}\n\n{report}")
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return f1

resultados_globales = {}
for current_dataset in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    study_name=f"study_name_{current_dataset}"
    study = optuna.create_study(direction='maximize',study_name=study_name)
    study.optimize(objective, n_trials=25)
    
    resultados_globales[current_dataset] = {
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_trial': study.best_trial.number
    }

informe_final = "Informe: Baseline MLP\n"
for ds, met in resultados_globales.items():
    informe_final += f"Dataset: {ds} | F1: {met['best_f1_macro']:.4f}\n"

with open(f"{log_folder}/resumen_resultados.txt", 'w') as f: f.write(informe_final)
print(informe_final)

[I 2026-03-19 00:27:22,286] A new study created in memory with name: study_name_none
[I 2026-03-19 00:29:00,420] Trial 0 finished with value: 0.7875877045334373 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 896, 'lr': 0.0008094278068835798, 'gamma': 3.9511772794080198}. Best is trial 0 with value: 0.7875877045334373.
[I 2026-03-19 00:29:51,644] Trial 1 finished with value: 0.8758295488151338 and parameters: {'batch_size': 8192, 'n_layers': 2, 'n_units': 512, 'lr': 0.00010559157893203167, 'gamma': 1.8961889877390956}. Best is trial 1 with value: 0.8758295488151338.
[I 2026-03-19 00:31:28,058] Trial 2 finished with value: 0.8720369661528558 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 512, 'lr': 0.00019005834038390618, 'gamma': 2.3660176289352606}. Best is trial 1 with value: 0.8758295488151338.
[I 2026-03-19 00:33:04,219] Trial 3 finished with value: 0.8758509405673457 and parameters: {'batch_size': 4096, 'n_layers': 2, 'n_units': 896, 'lr': 0.00068973

Informe: Baseline MLP
Dataset: none | F1: 0.8838
Dataset: smote_tomek | F1: 0.8858
Dataset: smote | F1: 0.8928
Dataset: smote_enn | F1: 0.8175



In [15]:
import os
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import warnings
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import QuantileTransformer

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")
device = torch.device("cuda")

log_folder = "Logs_MLP_Baseline_3"
os.makedirs(log_folder, exist_ok=True)

class ResidualBlock(nn.Module):
    def __init__(self, units, dropout_rate):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(units, units),
            nn.BatchNorm1d(units),
            nn.LeakyReLU(0.01),
            nn.Dropout(dropout_rate)
        )
    def forward(self, x):
        return x + self.block(x)

class TabularResNet(nn.Module):
    def __init__(self, input_dim, n_blocks, n_units, dropout_rate):
        super().__init__()
        self.first_layer = nn.Sequential(
            nn.Linear(input_dim, n_units),
            nn.BatchNorm1d(n_units),
            nn.LeakyReLU(0.01)
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(n_units, dropout_rate) for _ in range(n_blocks)
        ])
        self.final_output = nn.Linear(n_units, 13)

    def forward(self, x):
        x = self.first_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.final_output(x)

def objective(trial):
    global X_train_vram, y_train_vram, X_val_vram, y_val_cpu, current_dataset
    
    batch_size = trial.suggest_categorical('batch_size', [8192, 16384])
    n_blocks = trial.suggest_int('n_blocks', 3, 6)
    n_units = trial.suggest_int('n_units', 512, 1024, step=128)
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    gamma_opt = trial.suggest_float('gamma', 2.0, 4.0)
    
    model = TabularResNet(X_train_vram.shape[1], n_blocks, n_units, 0.2).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    
    criterion = FocalLoss(gamma=gamma_opt)

    num_samples = X_train_vram.shape[0]
    for epoch in range(50):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = X_train_vram[indices], y_train_vram[indices]
            
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_val_vram)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        f1 = f1_score(y_val_cpu, preds, average='macro')
    
    save_confusion_matrix_mlp(log_folder, y_val_cpu, preds, trial.number, current_dataset)
    
    report = classification_report(y_val_cpu, preds, zero_division=0)
    with open(f"{log_folder}/resnet_{current_dataset}_trial_{trial.number}.log", "w") as f:
        f.write(f"F1 Macro: {f1:.4f}\nParams: {trial.params}\n\n{report}")
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return f1

resultados_globales = {}

for ds_name in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    print(f"\nIniciando tests con Residual Networks para: {ds_name}")
    
    current_dataset = ds_name 
    
    X_train_raw, y_train_raw = train_datasets_grouped_2[ds_name]
    X_val_raw = X_val_imp_grouped_2.values
    qt = QuantileTransformer(n_quantiles=2000, output_distribution='normal', random_state=42)
    X_train_gaussian = qt.fit_transform(X_train_raw)
    X_val_gaussian = qt.transform(X_val_raw)
    X_train_vram = torch.tensor(X_train_gaussian.astype(np.float32)).to(device)
    y_train_vram = torch.tensor(np.array(y_train_raw, dtype=np.int64)).to(device)
    X_val_vram = torch.tensor(X_val_gaussian.astype(np.float32)).to(device)
    y_val_cpu = np.array(y_val_grouped_2, dtype=np.int64)

    study_name = f"study_resnet_{ds_name}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective, n_trials=30)
    
    resultados_globales[ds_name] = {
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_trial': study.best_trial.number
    }

    del X_train_vram, y_train_vram, X_val_vram
    gc.collect()
    torch.cuda.empty_cache()

informe_final = "Informe: ResNet Baseline 3\n"
for ds, met in resultados_globales.items():
    informe_final += f"Dataset: {ds:12} | F1: {met['best_f1_macro']:.4f} | Trial: {met['best_trial']}\n"

print(informe_final)
with open(f"{log_folder}/resumen_final.txt", 'w') as f:
    f.write(informe_final)


Iniciando tests con Residual Networks para: none


[I 2026-03-19 17:49:52,860] A new study created in memory with name: study_resnet_none
[I 2026-03-19 17:50:50,413] Trial 0 finished with value: 0.9299719555907282 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 896, 'lr': 0.00010146016095789741, 'gamma': 2.453703206312955}. Best is trial 0 with value: 0.9299719555907282.
[I 2026-03-19 17:51:35,510] Trial 1 finished with value: 0.9034910878812858 and parameters: {'batch_size': 16384, 'n_blocks': 5, 'n_units': 512, 'lr': 0.0008225565608894831, 'gamma': 3.1769672971603473}. Best is trial 0 with value: 0.9299719555907282.
[I 2026-03-19 17:52:41,952] Trial 2 finished with value: 0.8294341492474548 and parameters: {'batch_size': 8192, 'n_blocks': 3, 'n_units': 1024, 'lr': 0.0005942330259807194, 'gamma': 2.204033879601298}. Best is trial 0 with value: 0.9299719555907282.
[I 2026-03-19 17:53:39,432] Trial 3 finished with value: 0.923763572803506 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 896, 'lr': 0.000158


Iniciando tests con Residual Networks para: smote_tomek


[I 2026-03-19 18:22:12,419] A new study created in memory with name: study_resnet_smote_tomek
[I 2026-03-19 18:23:29,292] Trial 0 finished with value: 0.8333488300590424 and parameters: {'batch_size': 8192, 'n_blocks': 4, 'n_units': 512, 'lr': 0.00033347875296680675, 'gamma': 2.3940467177766287}. Best is trial 0 with value: 0.8333488300590424.
[I 2026-03-19 18:24:36,619] Trial 1 finished with value: 0.8440748897624312 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 1024, 'lr': 0.0007337366311744545, 'gamma': 2.9703776171680483}. Best is trial 1 with value: 0.8440748897624312.
[I 2026-03-19 18:25:27,316] Trial 2 finished with value: 0.9160118670747066 and parameters: {'batch_size': 16384, 'n_blocks': 6, 'n_units': 512, 'lr': 0.00010566609241531629, 'gamma': 3.8347478868676905}. Best is trial 2 with value: 0.9160118670747066.
[I 2026-03-19 18:26:57,812] Trial 3 finished with value: 0.8746101716846881 and parameters: {'batch_size': 8192, 'n_blocks': 5, 'n_units': 768, 'lr'


Iniciando tests con Residual Networks para: smote


[I 2026-03-19 19:01:23,773] A new study created in memory with name: study_resnet_smote
[I 2026-03-19 19:02:42,367] Trial 0 finished with value: 0.8921105777241146 and parameters: {'batch_size': 8192, 'n_blocks': 4, 'n_units': 640, 'lr': 0.0005019391664446588, 'gamma': 2.754421799273003}. Best is trial 0 with value: 0.8921105777241146.
[I 2026-03-19 19:03:21,289] Trial 1 finished with value: 0.8289459395506088 and parameters: {'batch_size': 16384, 'n_blocks': 3, 'n_units': 512, 'lr': 0.0007122690547197967, 'gamma': 3.137982988172699}. Best is trial 0 with value: 0.8921105777241146.
[I 2026-03-19 19:04:39,623] Trial 2 finished with value: 0.8849822821994527 and parameters: {'batch_size': 8192, 'n_blocks': 4, 'n_units': 768, 'lr': 0.0001795961248459709, 'gamma': 3.7204333570102404}. Best is trial 0 with value: 0.8921105777241146.
[I 2026-03-19 19:06:08,142] Trial 3 finished with value: 0.9040247689607075 and parameters: {'batch_size': 8192, 'n_blocks': 5, 'n_units': 1024, 'lr': 0.0001950


Iniciando tests con Residual Networks para: smote_enn


[I 2026-03-19 19:38:45,923] A new study created in memory with name: study_resnet_smote_enn
[I 2026-03-19 19:39:31,439] Trial 0 finished with value: 0.8393952309356465 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 768, 'lr': 0.0005365537491488206, 'gamma': 3.744638248928697}. Best is trial 0 with value: 0.8393952309356465.
[I 2026-03-19 19:40:10,601] Trial 1 finished with value: 0.8536183887710921 and parameters: {'batch_size': 16384, 'n_blocks': 3, 'n_units': 768, 'lr': 0.00048373149076632487, 'gamma': 3.9173317997552495}. Best is trial 1 with value: 0.8536183887710921.
[I 2026-03-19 19:41:26,850] Trial 2 finished with value: 0.8201041695214977 and parameters: {'batch_size': 8192, 'n_blocks': 4, 'n_units': 512, 'lr': 0.00017971211187091878, 'gamma': 3.296767760363792}. Best is trial 1 with value: 0.8536183887710921.
[I 2026-03-19 19:42:34,349] Trial 3 finished with value: 0.8282982667732337 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 1024, 'lr': 0

Informe: ResNet Baseline 3
Dataset: none         | F1: 0.9431 | Trial: 18
Dataset: smote_tomek  | F1: 0.9349 | Trial: 23
Dataset: smote        | F1: 0.9485 | Trial: 15
Dataset: smote_enn    | F1: 0.8725 | Trial: 5



In [ ]:
#Pruebo de igual forma el uso de redes neuronales con Focal Loss sobre mi dataset sin agrupar para validar si asi consigo mejores resultados en capturar relaciones no lineales para ataques de la familia Web

In [13]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [14]:
#Carga de datos post oversampling
import joblib

X_train_none = joblib.load("Sets_Oversampling/X_train_none.joblib")
y_train_none = joblib.load("Sets_Oversampling/y_train_none.joblib")

X_train_smote = joblib.load("Sets_Oversampling/X_train_smote.joblib")
y_train_smote = joblib.load("Sets_Oversampling/y_train_smote.joblib")

X_train_smote_enn = joblib.load("Sets_Oversampling/X_train_smote_enn.joblib")
y_train_smote_enn = joblib.load("Sets_Oversampling/y_train_smote_enn.joblib")

X_train_smote_tomek = joblib.load("Sets_Oversampling/X_train_smote_tomek.joblib")
y_train_smote_tomek = joblib.load("Sets_Oversampling/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)

In [15]:
X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")


In [16]:
train_datasets = {
    'none': (X_train_none, y_train_none.values.ravel() if isinstance(y_train_none, pd.DataFrame) else y_train_none.ravel()),
    'smote_enn': (X_train_smote_enn, y_train_smote_enn.values.ravel() if isinstance(y_train_smote_enn, pd.DataFrame) else y_train_smote_enn.ravel()),
    'smote_tomek': (X_train_smote_tomek, y_train_smote_tomek.values.ravel() if isinstance(y_train_smote_tomek, pd.DataFrame) else y_train_smote_tomek.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [17]:
train_datasets_2 = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

In [18]:
#100 epocas con early stopping, probar en datasets sin agrupar y probar alguna otra arquitectura de red neuoranl

In [19]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, input, target):
        log_p = F.log_softmax(input, dim=1)
        p = torch.exp(log_p)

        n_classes = input.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(input)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        
        focal_weight = (1 - p) ** self.gamma
        
        loss = - (true_dist * focal_weight * log_p)
        
        return loss.sum(dim=1).mean()

In [19]:
import os
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gc
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import QuantileTransformer
from torch.optim.lr_scheduler import ReduceLROnPlateau

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")
device = torch.device("cuda")

log_folder = "Logs_MLP_Baseline_4"
os.makedirs(log_folder, exist_ok=True)

class ResidualBlock(nn.Module):
    def __init__(self, units, dropout_rate):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(units, units),
            nn.BatchNorm1d(units),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
    def forward(self, x):
        return x + self.block(x)

class TabularResNet(nn.Module):
    def __init__(self, input_dim, n_blocks, n_units, dropout_rate):
        super().__init__()
        self.first_layer = nn.Sequential(
            nn.Linear(input_dim, n_units),
            nn.BatchNorm1d(n_units),
            nn.GELU()
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(n_units, dropout_rate) for _ in range(n_blocks)
        ])
        self.final_output = nn.Linear(n_units, 13)

    def forward(self, x):
        x = self.first_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.final_output(x)

def objective(trial):
    global X_train_vram, y_train_vram, X_val_vram, y_val_cpu, current_dataset
    
    batch_size = trial.suggest_categorical('batch_size', [8192, 16384])
    n_blocks = trial.suggest_int('n_blocks', 4, 10)
    n_units = trial.suggest_int('n_units', 512, 1024, step=128)
    lr = trial.suggest_float('lr', 1e-4, 8e-4, log=True)
    
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
    w_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    
    gamma_opt = trial.suggest_float('gamma', 2.0, 3.5)
    l_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.1)
    
    model = TabularResNet(X_train_vram.shape[1], n_blocks, n_units, dropout_rate).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=w_decay)
    
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = FocalLoss(gamma=gamma_opt, smoothing=l_smoothing)

    best_f1_trial = 0
    best_preds = None
    patience_early_stopping = 12
    counter = 0

    num_samples = X_train_vram.shape[0]
    for epoch in range(150):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = X_train_vram[indices], y_train_vram[indices]
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_vram)
            val_preds = torch.argmax(val_logits, dim=1).cpu().numpy()
            epoch_f1 = f1_score(y_val_cpu, val_preds, average='macro')
        
        scheduler.step(epoch_f1)

        if epoch_f1 > best_f1_trial:
            best_f1_trial = epoch_f1
            best_preds = val_preds.copy()
            counter = 0
        else:
            counter += 1
        
        if counter >= patience_early_stopping:
            break

    save_confusion_matrix_mlp(log_folder, y_val_cpu, best_preds, trial.number, current_dataset)
    report = classification_report(y_val_cpu, best_preds, zero_division=0)
    
    with open(f"{log_folder}/resnet_{current_dataset}_trial_{trial.number}.log", "w") as f:
        f.write(f"F1 Macro (Best): {best_f1_trial:.4f}\nParams: {trial.params}\n\n{report}")
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_f1_trial

resultados_globales = {}

for ds_name in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    print(f"\nRed Neuronal Baseline 4 para: {ds_name}")
    current_dataset = ds_name 
    
    X_train_raw, y_train_raw = train_datasets_grouped_2[ds_name]
    X_val_raw = X_val_imp_grouped_2.values
    qt = QuantileTransformer(n_quantiles=2000, output_distribution='normal', random_state=42)
    X_train_gaussian = qt.fit_transform(X_train_raw)
    X_val_gaussian = qt.transform(X_val_raw)
    
    X_train_vram = torch.tensor(X_train_gaussian.astype(np.float32)).to(device)
    y_train_vram = torch.tensor(np.array(y_train_raw, dtype=np.int64)).to(device)
    X_val_vram = torch.tensor(X_val_gaussian.astype(np.float32)).to(device)
    y_val_cpu = np.array(y_val_grouped_2, dtype=np.int64)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    
    resultados_globales[ds_name] = {
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_trial': study.best_trial.number
    }

    del X_train_vram, y_train_vram, X_val_vram
    gc.collect(); torch.cuda.empty_cache()

informe_final = "Informe: Residual Network con algunas mejoras- Baseline 4\n"
for ds, met in resultados_globales.items():
    informe_final += f"Dataset: {ds:12} | F1: {met['best_f1_macro']:.4f} | Trial: {met['best_trial']}\n"

print(informe_final)
with open(f"{log_folder}/resumen_final.txt", 'w') as f:
    f.write(informe_final)


Red Neuronal Baseline 4 para: none


[I 2026-03-20 01:03:25,169] A new study created in memory with name: no-name-99870764-0b4e-4e71-bab3-8e1612b7ecbb
[I 2026-03-20 01:04:24,257] Trial 0 finished with value: 0.9075694333761317 and parameters: {'batch_size': 8192, 'n_blocks': 4, 'n_units': 768, 'lr': 0.0005980197071869921, 'dropout_rate': 0.20959541806912424, 'weight_decay': 3.550362230417652e-05, 'gamma': 3.482519859345868, 'label_smoothing': 0.08169572104557794}. Best is trial 0 with value: 0.9075694333761317.
[I 2026-03-20 01:05:05,147] Trial 1 finished with value: 0.8698137058992341 and parameters: {'batch_size': 16384, 'n_blocks': 8, 'n_units': 512, 'lr': 0.00013923757547877865, 'dropout_rate': 0.1560281932702213, 'weight_decay': 6.544279414924745e-06, 'gamma': 2.686588724488698, 'label_smoothing': 0.07770644239917335}. Best is trial 0 with value: 0.9075694333761317.
[I 2026-03-20 01:06:35,390] Trial 2 finished with value: 0.9231958680085977 and parameters: {'batch_size': 8192, 'n_blocks': 8, 'n_units': 768, 'lr': 0.0


Red Neuronal Baseline 4 para: smote_tomek


[I 2026-03-20 02:11:29,114] A new study created in memory with name: no-name-2db91c7c-8189-4bfa-b7a3-e65b7adad0a3
[I 2026-03-20 02:12:32,067] Trial 0 finished with value: 0.9006263945502806 and parameters: {'batch_size': 8192, 'n_blocks': 5, 'n_units': 640, 'lr': 0.0006289951174722333, 'dropout_rate': 0.1762936461903518, 'weight_decay': 6.234358850855858e-06, 'gamma': 2.3444658798863216, 'label_smoothing': 0.019973841748233436}. Best is trial 0 with value: 0.9006263945502806.
[I 2026-03-20 02:14:14,329] Trial 1 finished with value: 0.8990084318669797 and parameters: {'batch_size': 16384, 'n_blocks': 10, 'n_units': 896, 'lr': 0.0006542671534080582, 'dropout_rate': 0.13039582013715095, 'weight_decay': 3.788006524297559e-05, 'gamma': 3.160766375658522, 'label_smoothing': 0.00641980943825784}. Best is trial 0 with value: 0.9006263945502806.
[I 2026-03-20 02:15:06,606] Trial 2 finished with value: 0.8787143602768754 and parameters: {'batch_size': 16384, 'n_blocks': 8, 'n_units': 896, 'lr': 


Red Neuronal Baseline 4 para: smote


[I 2026-03-20 03:09:24,057] A new study created in memory with name: no-name-9af8af41-2fef-4172-8353-bd5823c245ec
[I 2026-03-20 03:11:15,434] Trial 0 finished with value: 0.9329989620896877 and parameters: {'batch_size': 8192, 'n_blocks': 7, 'n_units': 512, 'lr': 0.0002826819023492017, 'dropout_rate': 0.23985166672437594, 'weight_decay': 5.636173309895173e-05, 'gamma': 3.261442768990504, 'label_smoothing': 0.05899944733079215}. Best is trial 0 with value: 0.9329989620896877.
[I 2026-03-20 03:12:44,454] Trial 1 finished with value: 0.9388555925197941 and parameters: {'batch_size': 16384, 'n_blocks': 7, 'n_units': 768, 'lr': 0.0006630654121798789, 'dropout_rate': 0.27988980096458294, 'weight_decay': 2.601088772536537e-06, 'gamma': 2.980410286742204, 'label_smoothing': 0.03625824287853631}. Best is trial 1 with value: 0.9388555925197941.
[I 2026-03-20 03:13:43,908] Trial 2 finished with value: 0.9079218277068309 and parameters: {'batch_size': 16384, 'n_blocks': 9, 'n_units': 768, 'lr': 0.


Red Neuronal Baseline 4 para: smote_enn


[I 2026-03-20 04:22:49,520] A new study created in memory with name: no-name-39c223a9-85ed-4821-8fef-d1170ad36810
[I 2026-03-20 04:24:54,016] Trial 0 finished with value: 0.8648922638313489 and parameters: {'batch_size': 16384, 'n_blocks': 10, 'n_units': 1024, 'lr': 0.00023275284170586077, 'dropout_rate': 0.29062805167425776, 'weight_decay': 1.4121841303569738e-06, 'gamma': 2.6500912925004387, 'label_smoothing': 0.07101400919900026}. Best is trial 0 with value: 0.8648922638313489.
[I 2026-03-20 04:25:37,594] Trial 1 finished with value: 0.8643603063826989 and parameters: {'batch_size': 16384, 'n_blocks': 6, 'n_units': 512, 'lr': 0.0003051527340286984, 'dropout_rate': 0.25654399659845495, 'weight_decay': 2.0120484911949323e-05, 'gamma': 2.050259096066101, 'label_smoothing': 0.07386315540076496}. Best is trial 0 with value: 0.8648922638313489.
[I 2026-03-20 04:26:32,586] Trial 2 finished with value: 0.8626719211128412 and parameters: {'batch_size': 16384, 'n_blocks': 9, 'n_units': 640, '

Informe: Residual Network con algunas mejoras- Baseline 4
Dataset: none         | F1: 0.9480 | Trial: 37
Dataset: smote_tomek  | F1: 0.9338 | Trial: 6
Dataset: smote        | F1: 0.9461 | Trial: 21
Dataset: smote_enn    | F1: 0.8824 | Trial: 32



In [ ]:
import os
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gc
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import QuantileTransformer
from torch.optim.lr_scheduler import ReduceLROnPlateau

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")
device = torch.device("cuda")

log_folder = "Logs_MLP_Baseline_5"
os.makedirs(log_folder, exist_ok=True)

class ResidualBlock(nn.Module):
    def __init__(self, units, dropout_rate):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(units, units),
            nn.BatchNorm1d(units),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
    def forward(self, x):
        return x + self.block(x)

class TabularResNet(nn.Module):
    def __init__(self, input_dim, n_blocks, n_units, dropout_rate):
        super().__init__()
        self.first_layer = nn.Sequential(
            nn.Linear(input_dim, n_units),
            nn.BatchNorm1d(n_units),
            nn.GELU()
        )
        self.blocks = nn.ModuleList([
            ResidualBlock(n_units, dropout_rate) for _ in range(n_blocks)
        ])
        self.final_output = nn.Linear(n_units, 15)

    def forward(self, x):
        x = self.first_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.final_output(x)

def objective(trial):
    global X_train_vram, y_train_vram, X_val_vram, y_val_cpu, current_dataset
    
    batch_size = trial.suggest_categorical('batch_size', [8192, 16384])
    n_blocks = trial.suggest_int('n_blocks', 4, 10)
    n_units = trial.suggest_int('n_units', 512, 1024, step=128)
    lr = trial.suggest_float('lr', 1e-4, 8e-4, log=True)
    
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
    w_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    
    gamma_opt = trial.suggest_float('gamma', 2.0, 3.5)
    l_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.1)
    
    model = TabularResNet(X_train_vram.shape[1], n_blocks, n_units, dropout_rate).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=w_decay)
    
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = FocalLoss(gamma=gamma_opt, smoothing=l_smoothing)

    best_f1_trial = 0
    best_preds = None
    patience_early_stopping = 12
    counter = 0

    num_samples = X_train_vram.shape[0]
    for epoch in range(100):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = X_train_vram[indices], y_train_vram[indices]
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_vram)
            val_preds = torch.argmax(val_logits, dim=1).cpu().numpy()
            epoch_f1 = f1_score(y_val_cpu, val_preds, average='macro')
        
        scheduler.step(epoch_f1)

        if epoch_f1 > best_f1_trial:
            best_f1_trial = epoch_f1
            best_preds = val_preds.copy()
            counter = 0
        else:
            counter += 1
        
        if counter >= patience_early_stopping:
            break

    save_confusion_matrix_mlp(log_folder, y_val_cpu, best_preds, trial.number, current_dataset)
    report = classification_report(y_val_cpu, best_preds, zero_division=0)
    
    with open(f"{log_folder}/resnet_{current_dataset}_trial_{trial.number}.log", "w") as f:
        f.write(f"F1 Macro (Best): {best_f1_trial:.4f}\nParams: {trial.params}\n\n{report}")
    
    del model; gc.collect(); torch.cuda.empty_cache()
    return best_f1_trial

resultados_globales = {}

for ds_name in ['none', 'smote_tomek', 'smote', 'smote_enn']:
    print(f"\nRed Neuronal Baseline 5 para: {ds_name}")
    current_dataset = ds_name 
    
    X_train_raw, y_train_raw = train_datasets_2[ds_name]
    X_val_raw = X_val.values
    qt = QuantileTransformer(n_quantiles=2000, output_distribution='normal', random_state=42)
    X_train_gaussian = qt.fit_transform(X_train_raw)
    X_val_gaussian = qt.transform(X_val_raw)
    
    X_train_vram = torch.tensor(X_train_gaussian.astype(np.float32)).to(device)
    y_train_vram = torch.tensor(np.array(y_train_raw, dtype=np.int64)).to(device)
    X_val_vram = torch.tensor(X_val_gaussian.astype(np.float32)).to(device)
    y_val_cpu = np.array(y_val, dtype=np.int64)

    study_name=f"study_redNeuronal_NoAgrupado_{ds_name}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective, n_trials=25)
    
    resultados_globales[ds_name] = {
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_trial': study.best_trial.number
    }

    del X_train_vram, y_train_vram, X_val_vram
    gc.collect(); torch.cuda.empty_cache()

informe_final = "Informe: Residual Network con algunas mejoras- Baseline 4\n"
for ds, met in resultados_globales.items():
    informe_final += f"Dataset: {ds:12} | F1: {met['best_f1_macro']:.4f} | Trial: {met['best_trial']}\n"

print(informe_final)
with open(f"{log_folder}/resumen_final.txt", 'w') as f:
    f.write(informe_final)


Red Neuronal Baseline 5 para: none


[I 2026-03-20 18:03:26,029] A new study created in memory with name: study_redNeuronal_NoAgrupado_none
[I 2026-03-20 18:04:19,317] Trial 0 finished with value: 0.7864865982588196 and parameters: {'batch_size': 16384, 'n_blocks': 4, 'n_units': 896, 'lr': 0.0003734155676639029, 'dropout_rate': 0.24891660278496844, 'weight_decay': 7.517605721458219e-05, 'gamma': 2.0191447070178383, 'label_smoothing': 0.06017838000036783}. Best is trial 0 with value: 0.7864865982588196.
[I 2026-03-20 18:05:01,735] Trial 1 finished with value: 0.6612813476843024 and parameters: {'batch_size': 16384, 'n_blocks': 9, 'n_units': 768, 'lr': 0.00010019076336046692, 'dropout_rate': 0.23258672963065433, 'weight_decay': 2.914394265946307e-06, 'gamma': 2.8784871274318915, 'label_smoothing': 0.07489694876637541}. Best is trial 0 with value: 0.7864865982588196.
[I 2026-03-20 18:06:52,758] Trial 2 finished with value: 0.7876627320864825 and parameters: {'batch_size': 8192, 'n_blocks': 10, 'n_units': 768, 'lr': 0.0003432